In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from aequilibrae import Project
import logging
import sys
from aequilibrae import logger
import itertools
from copy import deepcopy
from src.build_graphs import build_graphs
from src.assignments import assignment
from src.format_matrices import format_matrices
import numpy as np
from aequilibrae.paths import TrafficAssignment, TrafficClass


pth = r"C:\Users\JakeMoss\models\Arkansas\model"

In [ ]:

proj = Project()
proj.open(pth)
net = proj.network

In [ ]:
zones = proj.zoning.data

In [ ]:
period = "am"

In [ ]:
proj_matrices = proj.matrices
carDemand = proj_matrices.get_matrix(f'{period.upper()}_omx')
carDemand.computational_view('AUTO')

lightTruckDemand = proj_matrices.get_matrix(f'{period.upper()}_omx')
lightTruckDemand.computational_view('COMMTRK')
lightTruckDemand.matrix_view = np.array(lightTruckDemand.matrix_view, np.float64)

heavyTruckDemand = proj_matrices.get_matrix(f'{period.upper()}_omx')
heavyTruckDemand.computational_view('HTRK')
heavyTruckDemand.matrix_view = np.array(heavyTruckDemand.matrix_view, np.float64)

In [ ]:
from quetzal.model import stepmodel

sm = stepmodel.StepModel()

In [ ]:
links = net.links.data
nodes = net.nodes.data

In [ ]:
links.value_counts("direction")

In [ ]:
5711 + 3092

In [ ]:
nodes

In [ ]:
zones

In [ ]:
# Link exclusions
with proj.db_connection as conn:
    sql = "select link_id from links where exclusionset IN ('TruckOnly', 'HOV2', 'HOV3')"
    exclude_from_passenger = [x[0] for x in conn.execute(sql).fetchall()]

    sql = "select link_id from links where exclusionset IN ('PassengerOnly', 'HOV2', 'HOV3')"
    exclude_from_truck = [x[0] for x in conn.execute(sql).fetchall()]

In [ ]:
excluded_links = set(links[(links.builtyear > 2010) | (links.removedyear < 2010) | (links.builtyear.isna() & links.removedyear.isna())].link_id.to_list() + links[(links.mode_code <10) | (links.mode_code >11)].link_id.to_list())

In [ ]:
links = links.loc[(~links.link_id.isin(excluded_links)) | (links["a_node"].isin(zones.zone_id) | links["b_node"].isin(zones.zone_id))]

In [ ]:
links["modes"] = links["modes"].apply(set)

p = (links["modes"].apply(lambda x: "c" in x)) & (links["link_id"].isin(exclude_from_passenger))
links.loc[p, "modes"] = links.loc[p, "modes"].apply(lambda x: x - {"c"})

p = (links["modes"].apply(lambda x: "T" in x)) & (links["link_id"].isin(exclude_from_truck))
links.loc[p, "modes"] = links.loc[p, "modes"].apply(lambda x: x - {"T"})

p = (links["modes"].apply(lambda x: "c" in x))
links.loc[p, "modes"] = links.loc[p, "modes"].apply(lambda x: x | {"L"})

links["modes"] = links["modes"].apply(lambda x: "".join(x))

In [ ]:
sm.zones = nodes.loc[nodes["is_centroid"] == 1, ["node_id", "is_centroid", "geometry"]].set_index("node_id")

In [ ]:
def flip_a_b_node(df):
    return df.assign(a_node=df["b_node"], b_node=df["a_node"])


columns = [
    "link_id", 
    "a_node", 
    "b_node", 
    "direction", 
    "link_type", 
    "modes", 
    "distance", 
    f"{period}_assncap_10_ab", 
    f"{period}_assncap_10_ba", 
    f"tt_{period}_10_ab", 
    f"tt_{period}_10_ba",
    "alpha_ab",
    "alpha_ba",
    "beta_ab",
    "beta_ba",
    "hov1tollcost",
    "mttollcost",
    "httollcost"
]

l = pd.concat(
    [
        links.loc[links["direction"] == 0, columns].assign(direction=1),
        flip_a_b_node(links.loc[links["direction"] == 0, columns]).assign(direction=-1),
        links.loc[links["direction"] == 1, columns],
        flip_a_b_node(links.loc[links["direction"] == -1, columns]),
    ],
    ignore_index=True,
    # axis=0
)

for field, field_ab, field_ba in [
    ("time", f"tt_{period}_10_ab", f"tt_{period}_10_ba"), 
    ("capacity", f"{period}_assncap_10_ab", f"{period}_assncap_10_ba"),
    ("alpha", f"alpha_ab", f"alpha_ba"),
    ("beta", f"beta_ab", f"beta_ba"),
]:
    l.loc[:, field] = float("NaN")
    l.loc[l["direction"] == 1, field] = l.loc[l["direction"] == 1, field_ab]
    l.loc[l["direction"] == -1, field] = l.loc[l["direction"] == -1, field_ba]



l = l.loc[l.a_node != l.b_node]
# l.loc[l.a_node == l.b_node, f"{period}_assncap_10_ba"] = 1.0

l.hov1tollcost = l.hov1tollcost.fillna(0)
l.mttollcost = l.mttollcost.fillna(0)
l.httollcost = l.httollcost.fillna(0)

l = l.rename(columns={
        "modes": "segments",
        "a_node": "a",
        "b_node": "b",
    }
)

l["segments"] = l["segments"].apply(set)
l["vdf"] = 1
l["time"] = l["time"].fillna(0.001)
l["capacity"] = l["capacity"].fillna(1)

l["alpha"] = l["alpha"].fillna(0.15)
l["beta"] = l["beta"].fillna(4.0)

l = l.set_index("link_id").drop(columns=[
    "am_assncap_10_ab",
    "am_assncap_10_ba", 
    "tt_am_10_ab",
    "tt_am_10_ba",
    "alpha_ab",
    "alpha_ba",
    "beta_ab",
    "beta_ba",
])
l.index.name = "index"

sm.road_links = l.copy()

In [ ]:
sm.road_links.isna().sum()

In [ ]:
# Quetzal can't handle multiple lins between the same two nodes, so we need to add a dummy node in between them and split the link into two

links_to_fix = sm.road_links[sm.road_links[["a", "b"]].duplicated()]

max_link_id = sm.road_links.index.max()
max_node_id = max(sm.road_links.a.max(), sm.road_links.b.max(), sm.zones.index.max())

middel_index = range(max_link_id + 1, max_link_id + 1 + len(links_to_fix))
middel_node = range(max_node_id + 1, max_node_id + 1 + len(links_to_fix))

sm.road_links = pd.concat(
    [
        sm.road_links.drop(links_to_fix.index),
        links_to_fix.assign(b=middel_node),
        links_to_fix.assign(a=middel_node, b=links_to_fix.a, distance=0, time=0, hov1tollcost=0, mttollcost=0, httollcost=0, index=middel_index).set_index("index"),
    ]
)

In [ ]:
l[l["a"].isin(sm.zones.index) | l["b"].isin(sm.zones.index)]

In [ ]:
sm.zone_to_road = l[l["a"].isin(sm.zones.index) | l["b"].isin(sm.zones.index)]
sm.zone_to_road["direction"] = sm.zone_to_road["direction"].map({1: "access", -1: "egress"})

In [ ]:
sm.road_links = sm.road_links.drop(columns=["direction"])

In [ ]:
sm.road_nodes = nodes.loc[nodes["is_centroid"] == 0, ["node_id", "is_centroid", "geometry"]].set_index("node_id")

In [ ]:
sm.volumes = pd.DataFrame(
    {
        "origin": np.tile(sm.zones.index, len(sm.zones)),
        "destination": np.repeat(sm.zones.index, len(sm.zones)),
        "c": heavyTruckDemand.matrix_view.flatten() * 1.0,
        "L": lightTruckDemand.matrix_view.flatten() * 1.5,
        "T": heavyTruckDemand.matrix_view.flatten() * 2.5,
    }
)

In [ ]:
tolerance = 1e-5
segments = ["c", "L", "T"]
num_threads = 8

cost_functions = {
    'c': f'jam_time + hov1tollcost + time * 0.2',
    'L': f'jam_time + mttollcost + time * 0.5',
    'T': f'jam_time + httollcost + time * 1.0',
}

vdf = {
    1: f'time * (1 + alpha * (flow / capacity) ** beta)',
}

In [ ]:
del links, nodes, zones, carDemand, lightTruckDemand, heavyTruckDemand

In [ ]:
# test
sm.step_road_pathfinder(
    method='bfw',
    maxiters=1,
    tolerance=tolerance,
    segments=segments,
    cost_functions=cost_functions,
    vdf=vdf,
    turn_penalties=None,
    num_cores=num_threads,
    return_car_los=False,
    log=True,
)